# Hourly Data Aggregation from Minute-Level Cache

This notebook converts minute-level OHLCV data from `pickle_month_cache` into hourly aggregated data.

**Input**: `/home/craz/crypto/crypto-data/pickle_month_cache/usdt_data_YYYY-MM.pkl` (minute-level)

**Output**: `/home/craz/crypto/crypto-data/pickle_hour_cache/usdt_data_YYYY-MM.pkl` (hourly)

**Data Structure**: MultiIndex (open_time, symbol) with columns ['open', 'high', 'low', 'close', 'volume']

In [2]:
import pandas as pd
import os
import glob
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Configuration
INPUT_PATH = '/home/craz/crypto/crypto-data/pickle_month_cache/'
OUTPUT_PATH = '/home/craz/crypto/crypto-data/pickle_hour_cache/'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Output directory created: {os.path.exists(OUTPUT_PATH)}")

Input path: /home/craz/crypto/crypto-data/pickle_month_cache/
Output path: /home/craz/crypto/crypto-data/pickle_hour_cache/
Output directory created: True


In [4]:
def get_available_month_files():
    """Get all available monthly pickle files"""
    files = glob.glob(os.path.join(INPUT_PATH, "usdt_data_*.pkl"))
    files.sort()
    return files

month_files = get_available_month_files()
print(f"Found {len(month_files)} monthly pickle files")
if month_files:
    print(f"First: {os.path.basename(month_files[0])}")
    print(f"Last: {os.path.basename(month_files[-1])}")

Found 70 monthly pickle files
First: usdt_data_2020-01.pkl
Last: usdt_data_2025-10.pkl


In [5]:
def load_minute_data(file_path):
    """Load minute-level data from pickle file"""
    try:
        # Try pandas method first
        df = pd.read_pickle(file_path)
        return df
    except Exception as e:
        print(f"Pandas read_pickle failed: {e}")
        # Fallback to standard pickle
        try:
            with open(file_path, 'rb') as f:
                df = pickle.load(f)
            return df
        except Exception as e2:
            print(f"Standard pickle.load failed: {e2}")
            return None

In [6]:
def resample_to_hourly(minute_df):
    """
    Resample minute-level data to hourly data.
    
    Args:
        minute_df: DataFrame with MultiIndex (open_time, symbol)
    
    Returns:
        DataFrame with hourly data, same MultiIndex structure
    """
    # Reset index to work with groupby
    df = minute_df.reset_index()
    
    # Ensure open_time is datetime
    df['open_time'] = pd.to_datetime(df['open_time'])
    
    # Create hour column for grouping
    df['hour'] = df['open_time'].dt.floor('H')
    
    # Group by hour and symbol, then aggregate OHLCV
    hourly_data = df.groupby(['hour', 'symbol']).agg({
        'open': 'first',   # First open price of the hour
        'high': 'max',     # Maximum high price in the hour
        'low': 'min',      # Minimum low price in the hour
        'close': 'last',   # Last close price of the hour
        'volume': 'sum'    # Total volume in the hour
    }).reset_index()
    
    # Rename hour back to open_time
    hourly_data = hourly_data.rename(columns={'hour': 'open_time'})
    
    # Set MultiIndex
    hourly_data = hourly_data.set_index(['open_time', 'symbol'])
    hourly_data = hourly_data.sort_index()
    
    return hourly_data

In [7]:
def save_hourly_data(hourly_df, year_month):
    """Save hourly data to pickle file"""
    output_file = os.path.join(OUTPUT_PATH, f"usdt_data_{year_month}.pkl")
    
    try:
        # Use pandas to_pickle for consistency
        hourly_df.to_pickle(output_file)
        return output_file
    except Exception as e:
        print(f"Error saving with pandas: {e}")
        # Fallback to standard pickle
        try:
            with open(output_file, 'wb') as f:
                pickle.dump(hourly_df, f)
            return output_file
        except Exception as e2:
            print(f"Error saving with pickle: {e2}")
            return None

In [8]:
def process_monthly_file(file_path):
    """
    Process a single monthly file: load, resample to hourly, save.
    
    Returns:
        tuple: (success, year_month, minute_rows, hourly_rows, minute_size_mb, hourly_size_mb)
    """
    file_name = os.path.basename(file_path)
    year_month = file_name.replace('usdt_data_', '').replace('.pkl', '')
    
    # Get input file size
    minute_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    
    # Load minute data
    minute_df = load_minute_data(file_path)
    if minute_df is None:
        print(f"Failed to load {file_name}")
        return False, year_month, 0, 0, minute_size_mb, 0
    
    minute_rows = len(minute_df)
    minute_symbols = len(minute_df.index.get_level_values('symbol').unique())
    
    # Resample to hourly
    hourly_df = resample_to_hourly(minute_df)
    hourly_rows = len(hourly_df)
    hourly_symbols = len(hourly_df.index.get_level_values('symbol').unique())
    
    # Save hourly data
    output_file = save_hourly_data(hourly_df, year_month)
    if output_file is None:
        print(f"Failed to save {year_month}")
        return False, year_month, minute_rows, hourly_rows, minute_size_mb, 0
    
    # Get output file size
    hourly_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    
    # Calculate compression ratio
    compression_ratio = (1 - hourly_size_mb / minute_size_mb) * 100 if minute_size_mb > 0 else 0
    
    print(f"✅ {year_month}: {minute_rows:,} → {hourly_rows:,} rows | "
          f"{minute_symbols} symbols | {minute_size_mb:.1f} MB → {hourly_size_mb:.1f} MB "
          f"({compression_ratio:.1f}% reduction)")
    
    return True, year_month, minute_rows, hourly_rows, minute_size_mb, hourly_size_mb

## Test with One Month

First, let's test the conversion process with a single month to verify it works correctly.

In [9]:
# Test with the first available month
if month_files:
    test_file = month_files[0]
    print(f"Testing with: {os.path.basename(test_file)}\n")
    
    success, year_month, min_rows, hour_rows, min_size, hour_size = process_monthly_file(test_file)
    
    if success:
        print(f"\n✅ Test successful!")
        print(f"Minute data: {min_rows:,} rows, {min_size:.1f} MB")
        print(f"Hourly data: {hour_rows:,} rows, {hour_size:.1f} MB")
        print(f"Ratio: {min_rows/hour_rows:.1f}x reduction in rows")
    else:
        print("\n❌ Test failed!")
else:
    print("No monthly files found to test!")

Testing with: usdt_data_2020-01.pkl

✅ 2020-01: 285,560 → 4,760 rows | 10 symbols | 12.6 MB → 0.2 MB (98.4% reduction)

✅ Test successful!
Minute data: 285,560 rows, 12.6 MB
Hourly data: 4,760 rows, 0.2 MB
Ratio: 60.0x reduction in rows


## Verify Test Output

Load the generated hourly file and verify its structure.

In [10]:
# Load and verify the test output
if month_files:
    test_year_month = os.path.basename(month_files[0]).replace('usdt_data_', '').replace('.pkl', '')
    test_output_file = os.path.join(OUTPUT_PATH, f"usdt_data_{test_year_month}.pkl")
    
    if os.path.exists(test_output_file):
        print(f"Loading test output: {os.path.basename(test_output_file)}\n")
        
        test_hourly = pd.read_pickle(test_output_file)
        
        print(f"Shape: {test_hourly.shape}")
        print(f"Index levels: {test_hourly.index.names}")
        print(f"Columns: {test_hourly.columns.tolist()}")
        print(f"\nDate range: {test_hourly.index.get_level_values('open_time').min()} to "
              f"{test_hourly.index.get_level_values('open_time').max()}")
        print(f"Unique symbols: {len(test_hourly.index.get_level_values('symbol').unique())}")
        print(f"Symbol list: {sorted(test_hourly.index.get_level_values('symbol').unique().tolist())}")
        
        print(f"\nFirst 10 rows:")
        print(test_hourly.head(10))
        
        print(f"\nData types:")
        print(test_hourly.dtypes)
        
        print(f"\nBasic statistics:")
        print(test_hourly.describe())
    else:
        print(f"Test output file not found: {test_output_file}")

Loading test output: usdt_data_2020-01.pkl

Shape: (4760, 5)
Index levels: ['open_time', 'symbol']
Columns: ['open', 'high', 'low', 'close', 'volume']

Date range: 2020-01-01 00:00:00 to 2020-01-31 23:00:00
Unique symbols: 10
Symbol list: ['ADAUSDT', 'BCHUSDT', 'BTCUSDT', 'ETCUSDT', 'ETHUSDT', 'LINKUSDT', 'LTCUSDT', 'TRXUSDT', 'XLMUSDT', 'XRPUSDT']

First 10 rows:
                                open     high      low    close     volume
open_time           symbol                                                
2020-01-01 00:00:00 BCHUSDT   206.58   207.92   206.15   206.82  13171.038
                    BTCUSDT  7191.80  7221.75  7184.46  7207.71   5079.656
                    ETHUSDT   129.61   130.51   129.06   130.01  41335.565
2020-01-01 01:00:00 BCHUSDT   206.82   207.28   204.15   204.28  19502.176
                    BTCUSDT  7207.78  7208.14  7146.73  7163.61   7237.620
                    ETHUSDT   130.01   130.08   128.63   129.32  43083.500
2020-01-01 02:00:00 BCHUSDT   204

## Batch Processing - All Months

Now process all available monthly files.

In [11]:
print(f"Starting batch processing for {len(month_files)} months...\n")
print("=" * 80)

results = []
total_minute_rows = 0
total_hourly_rows = 0
total_minute_size = 0
total_hourly_size = 0
failed_months = []

for file_path in tqdm(month_files, desc="Processing months"):
    success, year_month, min_rows, hour_rows, min_size, hour_size = process_monthly_file(file_path)
    
    if success:
        results.append({
            'year_month': year_month,
            'minute_rows': min_rows,
            'hourly_rows': hour_rows,
            'minute_size_mb': min_size,
            'hourly_size_mb': hour_size
        })
        total_minute_rows += min_rows
        total_hourly_rows += hour_rows
        total_minute_size += min_size
        total_hourly_size += hour_size
    else:
        failed_months.append(year_month)

print("\n" + "=" * 80)
print("\n=== BATCH PROCESSING COMPLETE ===")
print(f"Successfully processed: {len(results)} months")
print(f"Total minute rows: {total_minute_rows:,}")
print(f"Total hourly rows: {total_hourly_rows:,}")
print(f"Total minute size: {total_minute_size:,.1f} MB ({total_minute_size/1024:.2f} GB)")
print(f"Total hourly size: {total_hourly_size:,.1f} MB ({total_hourly_size/1024:.2f} GB)")
print(f"Total size reduction: {(1 - total_hourly_size/total_minute_size)*100:.1f}%")

if failed_months:
    print(f"\n⚠️  Failed months: {failed_months}")
else:
    print("\n✅ All months processed successfully!")

Starting batch processing for 70 months...



Processing months:   0%|          | 0/70 [00:00<?, ?it/s]

✅ 2020-01: 285,560 → 4,760 rows | 10 symbols | 12.6 MB → 0.2 MB (98.4% reduction)


Processing months:   3%|▎         | 2/70 [00:00<00:07,  9.21it/s]

✅ 2020-02: 760,470 → 12,676 rows | 23 symbols | 33.0 MB → 0.5 MB (98.4% reduction)


...

✅ 2025-08: 22,851,015 → 380,859 rows | 522 symbols | 1002.8 MB → 16.0 MB (98.4% reduction)


Processing months:  99%|█████████▊| 69/70 [01:35<00:03,  3.31s/it]

✅ 2025-09: 23,161,047 → 386,028 rows | 554 symbols | 1016.4 MB → 16.2 MB (98.4% reduction)


Processing months: 100%|██████████| 70/70 [01:35<00:00,  1.36s/it]

✅ 2025-10: 2,401,734 → 40,030 rows | 558 symbols | 100.8 MB → 1.6 MB (98.4% reduction)


=== BATCH PROCESSING COMPLETE ===
Successfully processed: 70 months
Total minute rows: 559,615,266
Total hourly rows: 9,327,073
Total minute size: 24,486.7 MB (23.91 GB)
Total hourly size: 390.6 MB (0.38 GB)
Total size reduction: 98.4%

✅ All months processed successfully!


## Processing Summary Table

In [12]:
# Create summary DataFrame
if results:
    summary_df = pd.DataFrame(results)
    summary_df['row_reduction_pct'] = (1 - summary_df['hourly_rows'] / summary_df['minute_rows']) * 100
    summary_df['size_reduction_pct'] = (1 - summary_df['hourly_size_mb'] / summary_df['minute_size_mb']) * 100
    
    print("\n=== PROCESSING SUMMARY ===")
    print(f"\nFirst 10 months:")
    print(summary_df.head(10).to_string(index=False))
    
    print(f"\n\nLast 10 months:")
    print(summary_df.tail(10).to_string(index=False))
    
    print(f"\n\nOverall Statistics:")
    print(f"Average row reduction: {summary_df['row_reduction_pct'].mean():.2f}%")
    print(f"Average size reduction: {summary_df['size_reduction_pct'].mean():.2f}%")
    print(f"Total data processed: {summary_df['minute_rows'].sum():,} minute rows")
    print(f"Total hourly data: {summary_df['hourly_rows'].sum():,} hourly rows")
else:
    print("No results to display!")


=== PROCESSING SUMMARY ===

First 10 months:
year_month  minute_rows  hourly_rows  minute_size_mb  hourly_size_mb  row_reduction_pct  size_reduction_pct
   2020-01       285560         4760       12.596825        0.202191          98.333100           98.394902
   2020-02       760470        12676       32.955907        0.526573          98.333136           98.402189
   2020-03      1026720        17112       44.404090        0.708851          98.333333           98.403636
   2020-04       993600        16560       42.971747        0.686031          98.333333           98.403529
   2020-05      1032959        17216       44.671850        0.713127          98.333332           98.403632
   2020-06      1096502        18276       47.387883        0.756462          98.333245           98.403680
   2020-07      1391033        23184       60.038829        0.957981          98.333325           98.404397
   2020-08      1660677        27678       71.610754        1.142331          98.333330   

## List All Generated Files

In [ ]:
# List all generated hourly pickle files
hourly_files = glob.glob(os.path.join(OUTPUT_PATH, "usdt_data_*.pkl"))
hourly_files.sort()

print(f"\n=== GENERATED HOURLY FILES ===")
print(f"Total files: {len(hourly_files)}\n")

for i, file in enumerate(hourly_files, 1):
    file_size = os.path.getsize(file) / (1024*1024)  # MB
    file_name = os.path.basename(file)
    print(f"{i:2d}. {file_name:<30} {file_size:>8.1f} MB")

if hourly_files:
    total_size_mb = sum(os.path.getsize(f) / (1024*1024) for f in hourly_files)
    print(f"\nTotal hourly cache size: {total_size_mb:,.1f} MB ({total_size_mb/1024:.2f} GB)")

## Final Verification

Verify a few random months to ensure data integrity.

In [ ]:
import random

# Verify 3 random files
if len(hourly_files) >= 3:
    print("\n=== VERIFICATION - Random Sampling ===")
    sample_files = random.sample(hourly_files, min(3, len(hourly_files)))
    
    for sample_file in sample_files:
        file_name = os.path.basename(sample_file)
        print(f"\n📊 Verifying: {file_name}")
        
        try:
            df = pd.read_pickle(sample_file)
            symbols = df.index.get_level_values('symbol').unique()
            date_range = (
                df.index.get_level_values('open_time').min(),
                df.index.get_level_values('open_time').max()
            )
            
            print(f"  ✅ Loaded successfully")
            print(f"  - Shape: {df.shape}")
            print(f"  - Symbols: {len(symbols)}")
            print(f"  - Date range: {date_range[0]} to {date_range[1]}")
            print(f"  - Columns: {df.columns.tolist()}")
            print(f"  - Index: {df.index.names}")
            print(f"  - No null values: {not df.isnull().any().any()}")
            
        except Exception as e:
            print(f"  ❌ Error loading: {e}")
else:
    print("Not enough files for verification")

## Completion

🎉 Hourly data aggregation complete! The hourly cache is ready for use in backtesting and analysis.

In [ ]:
print("\n" + "="*80)
print("🎉 HOURLY DATA AGGREGATION COMPLETE!")
print("="*80)
print(f"\n📁 Output directory: {OUTPUT_PATH}")
print(f"📊 Total files generated: {len(hourly_files)}")
print(f"💾 Total size: {sum(os.path.getsize(f) for f in hourly_files) / (1024*1024*1024):.2f} GB")
print(f"\n✅ Data is ready for backtesting and analysis!")
print(f"\nTo load hourly data in your code:")
print(f"  df = pd.read_pickle('/home/craz/crypto/crypto-data/pickle_hour_cache/usdt_data_2025-01.pkl')")